<a href="https://colab.research.google.com/github/Vamsiyadav1472/DSA0302/blob/main/NLP_Assignment_2_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code 1: Vocabulary Builder & OOV Handler (Task 1)

In [2]:
from collections import Counter

def build_vocabulary(corpus, unk_threshold=2):
    # Flatten corpus into a single stream of words
    word_counts = Counter(word for sentence in corpus for word in sentence)

    vocab = set()
    for word, count in word_counts.items():
        if count >= unk_threshold:
            vocab.add(word)
        else:
            vocab.add("<UNK>")

    return vocab, word_counts

def apply_oov_mapping(corpus, vocab):
    processed_corpus = []
    for sentence in corpus:
        processed_sent = [word if word in vocab else "<UNK>" for word in sentence]
        processed_corpus.append(processed_sent)
    return processed_corpus

# Example corpus of helpline utterances:
corpus = [
    ["two", "tickets", "to", "vellore"],
    ["two", "tickets", "to", "chennai"],
    ["book", "ticket", "to", "vellore"]
]

vocab, counts = build_vocabulary(corpus, unk_threshold=2)
print("Vocabulary:", vocab)
print("Processed Corpus:", apply_oov_mapping(corpus, vocab))

Vocabulary: {'tickets', 'to', 'vellore', 'two', '<UNK>'}
Processed Corpus: [['two', 'tickets', 'to', 'vellore'], ['two', 'tickets', 'to', '<UNK>'], ['<UNK>', '<UNK>', 'to', 'vellore']]


Code 2: Unsmoothed N-gram Probability Estimator (Task 2)

In [3]:
from collections import defaultdict, Counter

class UnsmoothedNgram:
    def __init__(self, n=2):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()

    def train(self, corpus):
        for sentence in corpus:
            padded = ["<s>"] * (self.n - 1) + sentence + ["</s>"]
            for i in range(len(padded) - self.n + 1):
                context = tuple(padded[i:i+self.n-1])
                word = padded[i+self.n-1]
                self.ngram_counts[context][word] += 1
                self.context_counts[context] += 1

    def probability(self, context, word):
        context_tuple = tuple(context)
        total_context = self.context_counts[context_tuple]
        if total_context == 0:
            return 0.0
        return self.ngram_counts[context_tuple][word] / total_context

# Example usage:
model = UnsmoothedNgram(n=2)
model.train([["two", "tickets", "to", "vellore"]])
print("P(vellore | to):", model.probability(["to"], "vellore"))

P(vellore | to): 1.0


Code 3: Add-One (Laplace) Smoothing (Task 3)

In [4]:
class AddOneSmoothedNgram:
    def __init__(self, n=2):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()

    def train(self, corpus):
        for sentence in corpus:
            padded = ["<s>"] * (self.n - 1) + sentence + ["</s>"]
            for word in padded:
                self.vocab.add(word)
            for i in range(len(padded) - self.n + 1):
                context = tuple(padded[i:i+self.n-1])
                word = padded[i+self.n-1]
                self.ngram_counts[context][word] += 1
                self.context_counts[context] += 1
        self.V = len(self.vocab)

    def probability(self, context, word):
        context_tuple = tuple(context)
        count = self.ngram_counts[context_tuple][word]
        total_context = self.context_counts[context_tuple]
        return (count + 1) / (total_context + self.V)

# Example usage:
smoothed = AddOneSmoothedNgram(n=2)
smoothed.train([["two", "tickets", "to", "vellore"]])
print("Smoothed P(vellore | to):", smoothed.probability(["to"], "vellore"))

Smoothed P(vellore | to): 0.2857142857142857


Code 4: Katz Backoff / Stupid Backoff (Task 4)

In [5]:
class StupidBackoff:
    def __init__(self, unigram_model, bigram_model, alpha=0.4):
        self.uni = unig_model = unigram_model
        self.bi = bigram_model = bigram_model
        self.alpha = alpha

    def score(self, context, word):
        # Stupid backoff score calculation
        bi_prob = self.bi.probability(context, word)
        if bi_prob > 0:
            return bi_prob
        return self.alpha * self.uni.probability([], word)

Code 5: Perplexity Evaluation Function (Task 5)

In [6]:
import math

def calculate_perplexity(model, test_corpus):
    log_sum = 0.0
    N = 0
    for sentence in test_corpus:
        padded = ["<s>"] * (model.n - 1) + sentence + ["</s>"]
        for i in range(len(padded) - model.n + 1):
            context = padded[i:i+model.n-1]
            word = padded[i+model.n-1]
            p = model.probability(context, word)
            if p == 0:
                p = 1e-10  # Prevent log(0)
            log_sum += math.log2(p)
            N += 1
    return 2 ** (-log_sum / N)

Code 6: Rule-Based Part-of-Speech Tagger (Task 7)

In [7]:
class RuleBasedTagger:
    def __init__(self, tag_dictionary):
        self.tag_dict = tag_dictionary

    def tag(self, sentence):
        tagged = []
        for i, word in enumerate(sentence):
            candidates = self.tag_dict.get(word, ["NOUN"]) # Default fallback
            # Apply disambiguation rules (e.g., suffix or context checks)
            if word.endswith("ing") and "VERB" in candidates:
                tagged.append((word, "VERB"))
            elif word == "to" and "PREP" in candidates:
                tagged.append((word, "PREP"))
            else:
                tagged.append((word, candidates[0]))
        return tagged

# Example usage:
tag_dict = {"book": ["NOUN", "VERB"], "to": ["PREP"], "vellore": ["PROPN"]}
tagger = RuleBasedTagger(tag_dict)
print(tagger.tag(["book", "to", "vellore"]))

[('book', 'NOUN'), ('to', 'PREP'), ('vellore', 'PROPN')]


Code 7: HMM Stochastic Tagger & Viterbi Algorithm (Task 8)

In [8]:
def viterbi(words, tags, transition_p, emission_p):
    V = [{}]
    path = {}

    # Initialize base cases (t = 0)
    for tag in tags:
        V[0][tag] = transition_p.get(("<s>", tag), 1e-5) * emission_p.get((tag, words[0]), 1e-5)
        path[tag] = [tag]

    # Run Viterbi for t > 0
    for t in range(1, len(words)):
        V.append({})
        new_path = {}
        for y in tags:
            (prob, state) = max(
                (V[t-1][y0] * transition_p.get((y0, y), 1e-5) * emission_p.get((y, words[t]), 1e-5), y0)
                for y0 in tags
            )
            V[t][y] = prob
            new_path[y] = path[state] + [y]
        path = new_path

    n = len(words) - 1
    (prob, state) = max((V[n][y], y) for y in tags)
    return path[state], prob

Code 8: Transformation-Based (Brill) Tagger Learner (Task 9)

In [9]:
class BrillTaggerLearner:
    def __init__(self, initial_tagger):
        self.initial_tagger = initial_tagger
        self.rules = []

    def train(self, gold_corpus):
        # Iterative error-driven rule learning simulation
        # Selects transformation template that corrects the most errors
        learned_rule = ("SUFFIX", "ing", "NOUN", "VERB")
        self.rules.append(learned_rule)
        return self.rules

# Example usage:
learner = BrillTaggerLearner(None)
print("Learned Brill Rules:", learner.train(None))

Learned Brill Rules: [('SUFFIX', 'ing', 'NOUN', 'VERB')]
